<a href="https://colab.research.google.com/github/jasminealyaa1212/DIP-final-project/blob/main/nyoba%20dulu%20deh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
import cv2
import numpy as np
import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Your specific Google Drive paths
video_path = '/content/drive/MyDrive/semester 3/Digital Image Processing/final project/video 2.mp4'

vehicle_xmls = {
    'car': '/content/drive/MyDrive/semester 3/Digital Image Processing/final project/cars.xml',
    'two_wheeler': '/content/drive/MyDrive/semester 3/Digital Image Processing/final project/two_wheeler.xml',
    'bus': '/content/drive/MyDrive/semester 3/Digital Image Processing/final project/bus.xml'
}

classifiers = {}
for label, xml_path in vehicle_xmls.items():
    if os.path.exists(xml_path):
        cascade = cv2.CascadeClassifier(xml_path)
        if not cascade.empty():
            classifiers[label] = cascade
            print(f"Loaded: {label}")

if not os.path.exists(video_path):
    raise FileNotFoundError(f"Video file not found at: {video_path}")

cap = cv2.VideoCapture(video_path)

# Variables from your fast code
trackers = {}
frame_count = 0

target_width = 1280
target_height = 720
frame_skip = 3  # Process 1 out of every 3 frames for speed

fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
processed_frames = []

print("Processing video frames...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Skip frames to keep CPU execution fast
    if frame_count % frame_skip != 0:
        frame_count += 1
        continue

    # Resize frame
    frame = cv2.resize(frame, (target_width, target_height))
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    current_centroids = []

    # Run all 3 vehicle detectors
    for label, cascade in classifiers.items():
        min_sz = (20, 20) if label == 'two_wheeler' else (30, 30)
        detections = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=3, minSize=min_sz)

        for (x, y, w, h) in detections:
            cx = int(x + w / 2)
            cy = int(y + h / 2)
            current_centroids.append((cx, cy, x, y, w, h, label))

    # Evaluate direction for each detected vehicle
    for (cx, cy, x, y, w, h, label) in current_centroids:
        is_violation = False

        # Check movement relative to nearby previous centroids
        for car_id, (px, py) in list(trackers.items()):
            if abs(cx - px) < 40 and abs(cy - py) < 50:
                dy = cy - py
                if dy < -2:  # Moving bottom-to-top against flow
                    is_violation = True
                break

        color = (0, 0, 255) if is_violation else (0, 255, 0)
        text = f"VIOLATION [{label}]" if is_violation else f"NORMAL [{label}]"

        # Draw bounding box and label
        cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
        cv2.putText(frame, text, (x, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    processed_frames.append(frame)
    frame_count += 1

cap.release()
print(f"Finished processing! Stored {len(processed_frames)} frames.")

Loaded: car
Loaded: two_wheeler
Loaded: bus
Processing video frames...
Finished processing! Stored 127 frames.


In [56]:
import cv2

output_video_path = '/content/drive/MyDrive/semester 3/Digital Image Processing/final project/annotated_output.mp4'

if 'processed_frames' not in globals() or len(processed_frames) == 0:
    raise RuntimeError("No processed frames found. Run Cell 1 first.")

print("Saving video...")

height, width, _ = processed_frames[0].shape
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# Adjust output FPS so saved video plays back at normal speed
out = cv2.VideoWriter(output_video_path, fourcc, max(1, fps // frame_skip), (width, height))

for frame in processed_frames:
    out.write(frame)

out.release()
print(f"Video saved to: {output_video_path}")

Saving video...
Video saved to: /content/drive/MyDrive/semester 3/Digital Image Processing/final project/annotated_output.mp4
